# Week 7 — Request a structured synthetic response

**Research task:** Predict one held-out `POLVIEWS` answer for a deidentified teaching profile, then separate schema validity from predictive evidence.

**Python introduced:** JSON Schema, required fields, numeric lists, `len(...)`, `sum(...)` and Boolean checks.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session07/session07_synthetic_representation.ipynb)

Colab supports the OpenRouter route only. Local JupyterLab or VS Code is canonical because it can also reach Ollama.

In [ ]:
# Colab setup: clone the public repository when running in Colab.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo = SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists():
        setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)], check=True)
    setup_os.chdir(setup_repo / 'workbook' / 'session07')
print('Working folder:', SetupPath.cwd())

## Load the course settings and SDKs

**Input:** installed Python packages, `config/course_models.json`, and—if it is not already set—the hidden OpenRouter key. **Operations:** `import` makes an installed tool available; `Path.cwd()` gives Python the current folder; the `while` block walks upward until it finds the course configuration; `json.loads(...)` turns the file's JSON text into a dictionary; square brackets retrieve the two model names. **Output:** `HOSTED_MODEL` and `LOCAL_MODEL` are strings. `getpass(...)` accepts the key without echoing it. The folder-search code is supplied setup and is not assessed.


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Choose a route and store one deidentified profile and survey item

`profile` is a dictionary containing public-use teaching attributes; `survey_item` is the exact question string. `held_out_human_answer` is stored separately and is not included in the prompt. This separation prevents the model from seeing the answer it is meant to predict.


In [ ]:
ROUTE = "ollama"  # change to "openrouter" if preferred
profile = {
    "age_group": "30–44",
    "education": "bachelor's degree",
    "region": "South",
    "party_identification": "independent",
}
question = "On a scale from 1 (very liberal) to 7 (very conservative), where would you place yourself?"
held_out_human_answer = 4
print(profile)
print(question)

## Define the exact output structure

The schema requires an integer category from 1 to 7 and a seven-number probability list. `required` says both fields must appear; `additionalProperties: False` prohibits extra fields. These rules concern readable structure and numeric bounds, not whether the prediction represents people well.


In [ ]:
schema = {
    "type": "object",
    "properties": {
        "predicted_category": {"type": "integer", "minimum": 1, "maximum": 7},
        "probabilities": {
            "type": "array", "items": {"type": "number"},
            "minItems": 7, "maxItems": 7,
        },
    },
    "required": ["predicted_category", "probabilities"],
    "additionalProperties": False,
}
prompt = (
    "Predict this deidentified respondent's answer. Return seven probabilities in "
    "order from 1 to 7. Profile: " + json.dumps(profile) + " Item: " + question
)
messages = [{"role": "user", "content": prompt}]

## Make the selected structured-output call

The prompt joins the profile and exact item into one message. OpenRouter places the schema in `response_format`; Ollama receives it through `format`. Both branches preserve the returned JSON text in `raw_json` before any field is extracted.


In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL, messages=messages, temperature=0,
            response_format={"type": "json_schema", "json_schema": {
                "name": "polviews_prediction", "strict": True, "schema": schema,
            }},
        )
    raw_output = response.choices[0].message.content
else:
    response = ollama.chat(
        model=LOCAL_MODEL, messages=messages, format=schema,
        options={"temperature": 0},
    )
    raw_output = response.message.content
print("Raw JSON text:", raw_output)

## Parse, check structure and only then compare with the held-out answer

Parsing creates a dictionary. `len(probabilities)` counts entries and `sum(probabilities)` adds them; comparisons produce Booleans checking seven values, an approximate total of one and values between zero and one. Only after those mechanical checks is `predicted_category` compared with the held-out human answer. One match is not population validation.


In [ ]:
prediction = json.loads(raw_output)
probabilities = prediction["probabilities"]
seven_values = len(probabilities) == 7
sum_is_close = abs(sum(probabilities) - 1.0) < 0.02
matches_human = prediction["predicted_category"] == held_out_human_answer
print("Predicted category:", prediction["predicted_category"])
print("Seven probabilities:", seven_values)
print("Sum close to one:", sum_is_close)
print("Held-out human answer:", held_out_human_answer)
print("Exact match:", matches_human)

# ONE CHANGE: change age_group to "18–29" and rerun.

## Methodological check

A valid schema means Python can retrieve the required fields. One exact match does not establish individual accuracy, group fidelity or population representation.
## Completion recording

Use one route, run the original profile, change only `age_group` and rerun. Explain every schema field, the raw JSON, the parsed list and all three checks. State why neither prediction establishes representativeness.

Explain every input and output aloud. Never show the shared key.